# NanoStat-CTV: supervised fine-tuning (arm S0) on a free T4 (Colab or Kaggle), then evaluate

**What this does.** LoRA-fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` on programmatic **gold typed plans** for the pilot **train**
split (`training/sft_data.py`: registry-preferred method, CPU-verified assumption verdicts, no teacher LLM). It then serves base + adapter
with vLLM on `127.0.0.1` and evaluates on the **same dev split** as the B1 baselines in `docs/LLM_B1_BASELINE.md`, with the same harness
and prompt. Splits are group-exclusive: no dev or test task appears in training.

**Kaggle:** set Accelerator to *GPU T4* (vLLM needs compute capability 7.0+, so not P100), turn Internet on, and use *Save Version → Save & Run All*. The run continues on Kaggle's servers if the browser disconnects; results land in `/kaggle/working/sft` and `sft_s0_results.zip`.

**Colab resilience.** Everything goes to Google Drive (`MyDrive/nanostat-ctv/sft`). Training checkpoints every 25 steps and resumes after a
disconnect. Evaluation traces are written per task and resume too. If the runtime drops, reconnect and **Run all** again.

**Runtime:** T4 GPU. Expect about 5 min of setup, 30–60 min of training (1 epoch, about 840 examples, not yet timed) and about 20 min of
evaluation. Report every result, including failures.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

# Works on Colab (results on Google Drive) and on Kaggle (results in /kaggle/working, kept as the run's output).
KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ROOT = "/kaggle/working" if KAGGLE else "/content"
REPO = f"{ROOT}/nanostat-ctv"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Anandharajan/nanostat-ctv {REPO}
%cd {REPO}
!git pull -q; git log -1 --format='%h %s'
!pip -q install -e '.[gpu]'
# Kaggle ships torchao 0.10; current PEFT refuses torchao < 0.16 while attaching LoRA. NanoStat does not use torchao.
!pip -q uninstall -y torchao 2>/dev/null || true

In [ ]:
import pathlib

if KAGGLE:
    OUT = "/kaggle/working/sft"  # Kaggle keeps /kaggle/working as the output of a "Save & Run All" run
else:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/nanostat-ctv/sft"
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
print("results ->", OUT)


def sh(cmd):
    # Run a shell step with its full output visible; stop the notebook if it fails (no silent failures).
    import subprocess

    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-6000:], flush=True)
    print(r.stderr[-6000:], flush=True)
    if r.returncode:
        raise RuntimeError(f"step failed with exit code {r.returncode}: {cmd}")

## 1 · Data: pilot factory, SFT examples from the train split only

In [ ]:
import hashlib

from training.sft_data import build_sft

sh("python -W ignore -m nanostat.cli factory --profile pilot --out data")
dev = pathlib.Path("data/tasks/dev.jsonl")
print("dev tasks", len(dev.read_text().splitlines()), "sha256", hashlib.sha256(dev.read_bytes()).hexdigest()[:16])
print(
    "SFT examples (train split):", build_sft(pathlib.Path("data"), pathlib.Path(OUT, "sft_train.jsonl"), split="train")
)

## 2 · Supervised fine-tuning (LoRA r=8, fp16 autocast, resumable)

In [ ]:
sh(
    f"python -m training.train_gpu sft --data data --sft {OUT}/sft_train.jsonl --out {OUT}/s0 --seed 42 --epochs 1 --save-steps 25"
)

In [ ]:
import json

log = json.loads(pathlib.Path(OUT, "s0", "train_log.json").read_text())
losses = [h["loss"] for h in log["log_history"] if "loss" in h]
print({k: v for k, v in log.items() if k != "log_history"})
print("loss first/last:", losses[:3], "...", losses[-3:])

## 3 · Serve base + adapter with vLLM, evaluate on the same dev split (prompt v3)

In [ ]:
sh("pip -q install vllm")

In [ ]:
import subprocess
import time
import urllib.request

BASE = "http://127.0.0.1:8000/v1"
log = open("vllm_s0.log", "w")
server = subprocess.Popen(
    [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "--served-model-name",
        "qwen2.5-coder-1.5b-instruct",
        "--enable-lora",
        "--max-lora-rank",
        "8",
        "--lora-modules",
        f"nanostat-s0={OUT}/s0/adapter",
        "--dtype",
        "half",
        "--max-model-len",
        "4096",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--seed",
        "0",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        ids = [m["id"] for m in json.load(urllib.request.urlopen(BASE + "/models", timeout=2))["data"]]
        print("serving:", ids)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited; see vllm_s0.log") from None
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not start in 15 min; see vllm_s0.log")

In [ ]:
from nanostat.prompting import PROMPT_VERSION, prompt_hash

PV = PROMPT_VERSION.removeprefix("typed-plan-")
print(PROMPT_VERSION, prompt_hash())
pathlib.Path(OUT, "eval").mkdir(exist_ok=True)

In [ ]:
for steps in (1, 3):
    sh(
        "python -W ignore -m nanostat.cli eval --arms B1 --llm-model nanostat-s0 --llm-base-url http://127.0.0.1:8000/v1 "
        f"--data data --split dev --max-steps {steps} --traces {OUT}/eval/s0_{PV}_dev_steps{steps}.jsonl "
        f"--out {OUT}/eval/s0_{PV}_dev_steps{steps}.json > /dev/null"
    )

In [ ]:
for f in sorted(pathlib.Path(OUT, "eval").glob("*.json")):
    r = json.loads(f.read_text())
    print(f.name, {k: v for k, v in r.items() if not isinstance(v, (dict, list))})

## 4 · Bring the result back (the adapter stays on Drive)

In [ ]:
!cp vllm_s0.log {OUT}/eval/ 2>/dev/null; (pip freeze | grep -E "^(vllm|torch|transformers|trl|peft)=="; nvidia-smi --query-gpu=name --format=csv,noheader; git log -1 --format=%h) > {OUT}/eval/versions_s0.txt
!cp {OUT}/s0/train_log.json {OUT}/eval/train_log_s0.json
!cd {OUT}/eval && rm -f {ROOT}/sft_s0_results.zip && zip -q {ROOT}/sft_s0_results.zip *
!ls -l {ROOT}/sft_s0_results.zip && sha256sum {ROOT}/sft_s0_results.zip
if not KAGGLE:
    from google.colab import files

    files.download(f"{ROOT}/sft_s0_results.zip")
else:
    print("Kaggle: download sft_s0_results.zip from the notebook's Output tab")